# L10 · SOD and SAGE-OPD

## Goal

**Estimated time:** 45 min · **Path:** full

- compute SOD step weights
- combine SAGE interventions
- preserve loss scale

### Current position: L09 → **L10** → L11

```text
Prompt/Data -> state source -> ... -> L10 -> ... -> fair evaluation
```

Alt text: The course map highlights L10 between its prerequisite and next lesson; every method remains connected to the same evaluation stage.

## Setup

In [1]:
LESSON_ID = "L10"
from pathlib import Path
import sys
import torch

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = Path.cwd().parents[1]
sys.path.insert(0, str(repo_root / "src"))

import opd_study
from opd_study.device import resolve_device
from opd_study.utils import seed_everything

seed_everything(42)
device_report = resolve_device("cpu")
print({"lesson": LESSON_ID, "opd_study": opd_study.__version__,
       "torch": torch.__version__, "device": device_report.selected,
       "profile": "toy", "network": "not required"})

{'lesson': 'L10', 'opd_study': '0.1.0.dev0', 'torch': '2.13.0', 'device': 'cpu', 'profile': 'toy', 'network': 'not required'}


## Steps

### 1/3 · 8–12 min

SOD attenuates unreliable regions using step divergence. SAGE-OPD multiplies intervention by teacher confidence, then normalizes to match dense-OPD loss scale.

Figure alt: labels and numbers remain readable without color.

### Core mechanics

SOD computes per-step teacher/student divergence and adjusts distillation weight around abrupt disagreements. Whether weights are detached changes the objective, not just implementation style. The mini implementation treats them as a stable curriculum signal and stops their gradients.

SAGE-OPD multiplies teacher-intervention need by teacher confidence, then normalizes token weights to the dense-OPD scale. All-zero intervention batches need an explicit skip/fallback policy. The mini judge is a token-agreement proxy, not a semantic-judge claim.

### Production implementation: why this design

SOD averages token KL by step, then broadcasts detached weights back to tokens. SAGE separates intervention labels, confidence, and normalization for ablation. The paper's separate GRPO term is omitted and labeled in metrics.

Production code: [`sod.py`](../../src/opd_study/algorithms/sod.py), [`sage_opd.py`](../../src/opd_study/algorithms/sage_opd.py).

In [2]:
import inspect
from opd_study.algorithms.sod import step_divergence_weights
from opd_study.algorithms.sage_opd import sage_opd_loss

objects_to_show = (step_divergence_weights, sage_opd_loss,)
for object_to_show in objects_to_show:
    source_lines = inspect.getsource(object_to_show).splitlines()
    print(f"\n# {object_to_show.__module__}.{object_to_show.__qualname__}")
    print("\n".join(source_lines[:80]))
    if len(source_lines) > 80:
        print(f"... {len(source_lines) - 80} more lines; open the linked source file")


# opd_study.algorithms.sod.step_divergence_weights
def step_divergence_weights(
    student_logits: Tensor,
    teacher_logits: Tensor,
    token_ids: Tensor,
    response_mask: Tensor,
    step_ids: Tensor,
    *,
    epsilon: float = 1e-6,
    delta: float = 0.2,
) -> tuple[Tensor, Tensor]:
    """Compute detached SOD ``d_k`` and cumulative-ratio ``w_k`` per token.

    ``d_k`` is the mean absolute sampled-token log-probability gap within a step.
    ``w_k = min(prod_{u<k}(d_u+eps)/(d_{u+1}+eps), 1+delta)``.
    """

    if student_logits.shape != teacher_logits.shape:
        raise ValueError("student and teacher logits must match")
    if token_ids.shape != response_mask.shape or token_ids.shape != step_ids.shape:
        raise ValueError("token_ids, response_mask and step_ids must match")
    if epsilon <= 0 or delta < 0:
        raise ValueError("epsilon must be positive and delta non-negative")
    student_log = torch.log_softmax(student_logits.float(), dim=-1)
    teacher_log 

### Alternatives and trade-offs

SOD and SAGE use different failure signals rather than being one competing recipe. Choose SOD when step divergence is a useful reliability proxy, SAGE when recoverability/teacher judgment can be queried. Combining them needs a new scale and double-counting audit.

### 2/3 · Run and observe

Predict before running: which invariant should you inspect first in L10's output? Write one sentence, then run.

In [3]:
from opd_study.algorithms.sod import step_divergence_weights
from opd_study.data import CharacterTokenizer, collate_multiturn_text, generate_tiny_arithmetic

tokenizer = CharacterTokenizer(); rows = generate_tiny_arithmetic(train_rows=2, validation_rows=1, test_rows=1).train
batch = collate_multiturn_text([(row.prompt, tuple(row.response.splitlines())) for row in rows], tokenizer)
shape = (*batch.token_ids.shape, tokenizer.vocab_size)
student_logits, teacher_logits = torch.randn(shape), torch.randn(shape)
divergence, sod_weights = step_divergence_weights(student_logits, teacher_logits,
    batch.token_ids, batch.response_mask, batch.step_ids)
print("SOD mean divergence/weight:", float(divergence[divergence > 0].mean()),
      float(sod_weights[sod_weights > 0].mean()))

SOD mean divergence/weight: 1.0277701616287231 1.0374205112457275


In [4]:
from opd_study.algorithms.sage_opd import sage_token_weights

number_of_turns = int(batch.turn_ids.max()) + 1
intervention = torch.tensor([[0.0, 0.5, 1.0]]).repeat(batch.token_ids.shape[0], 1)
sage_weights, confidence = sage_token_weights(teacher_logits, batch, intervention)
print("SAGE normalized token-weight sum:", float(sage_weights.sum()))
print("response token count:", int(batch.response_mask[:, 1:].sum()))
print("turn confidence row 0:", confidence[0].tolist())

SAGE normalized token-weight sum: 69.99999237060547
response token count: 70
turn confidence row 0: [0.10730810463428497, 0.09619998186826706, 0.0]


## Checks

In [5]:
assert not divergence.requires_grad and not sod_weights.requires_grad
assert abs(float(sage_weights.sum()) - int(batch.response_mask[:, 1:].sum())) < 1e-4
assert (sage_weights >= 0).all()
print("check passed: SOD weights detach; SAGE preserves dense-OPD loss scale")

check passed: SOD weights detach; SAGE preserves dense-OPD loss scale


**Exercise (10 min):** set all SAGE interventions to zero and one; compare skip/normalization and reconfirm SOD weights have no gradient.

<details><summary>Check</summary>Zero intervention is explicit, dense weights sum to response-token count, and SOD weights stay detached.</details>

## My recurring mistakes

### M1 — Omitting the SOD weight-gradient policy

- Wrong: leave detach behavior to incidental framework operations.
- Why: the objective and higher-order path change.
- Fix: specify and test detached curriculum weights.
- Related check: `test_sod_downweights_a_divergence_jump`

### M2 — Calling the mini SAGE proxy a semantic judge

- Wrong: report token agreement as teacher-judgment performance.
- Why: it does not measure semantic recoverability.
- Fix: label the proxy and unverified research judge.
- Related check: `test_sage_weights_normalize_and_skip`

## 60-second summary

1. compute SOD step weights
2. combine SAGE interventions
3. preserve loss scale

## Next Steps

Before the next notebook, rerun the assertions and record one prediction you revised.

### Sources

- [`sod`](https://arxiv.org/abs/2605.07725v3) · `2605.07725v3` · license `arXiv-non-exclusive-distribution-1.0` · [audited manifest](../../docs/sources.yml)
- [`sage_opd`](https://arxiv.org/abs/2606.19659v1) · `2606.19659v1` · license `CC-BY-4.0` · [audited manifest](../../docs/sources.yml)
- [`sod_official`](https://github.com/YoungZ365/SOD) · `110c4b8e843aee274d3cd648199569369ee2403e` · license `Apache-2.0` · [audited manifest](../../docs/sources.yml)